<a href="https://colab.research.google.com/github/Srinivas26k/Kaggle-Competitions/blob/main/predicting_road_accident_risk_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!unzip /content/playground-series-s5e10.zip

Archive:  /content/playground-series-s5e10.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings('ignore')

In [10]:
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

In [11]:
train_df.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [12]:
print("Dataset shapes:")
print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print(f"\nTrain columns: {train_df.columns.tolist()}")
print(f"\nFirst few rows:\n{train_df.head()}")

Dataset shapes:
Train: (517754, 14), Test: (172585, 13)

Train columns: ['id', 'road_type', 'num_lanes', 'curvature', 'speed_limit', 'lighting', 'weather', 'road_signs_present', 'public_road', 'time_of_day', 'holiday', 'school_season', 'num_reported_accidents', 'accident_risk']

First few rows:
   id road_type  num_lanes  curvature  speed_limit  lighting weather  \
0   0     urban          2       0.06           35  daylight   rainy   
1   1     urban          4       0.99           35  daylight   clear   
2   2     rural          4       0.63           70       dim   clear   
3   3   highway          4       0.07           35       dim   rainy   
4   4     rural          1       0.58           60  daylight   foggy   

   road_signs_present  public_road time_of_day  holiday  school_season  \
0               False         True   afternoon    False           True   
1                True        False     evening     True           True   
2               False         True     morning   

In [28]:
# Separate features and target
X_train = train_df.drop(["id","accident_risk"], axis=1)
y_train = train_df['accident_risk']
X_test = test_df.drop(["id"],axis=1)
test_ids = test_df["id"]

In [15]:
# Identify categorical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\nCategorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")


Categorical columns: ['road_type', 'lighting', 'weather', 'time_of_day']
Numerical columns: ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']


In [16]:
# Encode categorical variables
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

In [17]:
# Handle missing values
X_train = X_train.fillna(X_train.mean())
X_test = X_test.fillna(X_train.mean())

In [18]:
# Scale numerical features
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

In [22]:
# Split for validation
X_val_train, X_val_test, y_val_train, y_val_test = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)


In [23]:
print("\n" + "="*50)
print("Training Models")
print("="*50)

# Model 1: Gradient Boosting
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=7,
    min_samples_split=5,
    min_samples_leaf=2,
    subsample=0.8,
    random_state=42,
    verbose=0
)
gb_model.fit(X_val_train, y_val_train)
gb_pred_val = gb_model.predict(X_val_test)
gb_rmse = np.sqrt(np.mean((gb_pred_val - y_val_test)**2))
print(f"\nGradient Boosting RMSE: {gb_rmse:.6f}")




Training Models

Gradient Boosting RMSE: 0.056227


In [24]:
# Model 2: Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_val_train, y_val_train)
rf_pred_val = rf_model.predict(X_val_test)
rf_rmse = np.sqrt(np.mean((rf_pred_val - y_val_test)**2))
print(f"Random Forest RMSE: {rf_rmse:.6f}")

Random Forest RMSE: 0.056820


In [25]:
# Model 3: Ridge Regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_val_train, y_val_train)
ridge_pred_val = ridge_model.predict(X_val_test)
ridge_rmse = np.sqrt(np.mean((ridge_pred_val - y_val_test)**2))
print(f"Ridge Regression RMSE: {ridge_rmse:.6f}")

Ridge Regression RMSE: 0.088444


In [26]:
# Train on full training set for final predictions
print("\n" + "="*50)
print("Final Training on Full Dataset")
print("="*50)

gb_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
ridge_model.fit(X_train, y_train)

# Make predictions on test set
gb_test_pred = gb_model.predict(X_test)
rf_test_pred = rf_model.predict(X_test)
ridge_test_pred = ridge_model.predict(X_test)

# Ensemble: weighted average (GB gets more weight)
weights = np.array([0.5, 0.3, 0.2])  # GB, RF, Ridge
ensemble_pred = (weights[0] * gb_test_pred +
                 weights[1] * rf_test_pred +
                 weights[2] * ridge_test_pred)

# Clip predictions to [0, 1]
ensemble_pred = np.clip(ensemble_pred, 0, 1)

print(f"\nEnsemble predictions - Min: {ensemble_pred.min():.4f}, Max: {ensemble_pred.max():.4f}")
print(f"Mean: {ensemble_pred.mean():.4f}, Std: {ensemble_pred.std():.4f}")


Final Training on Full Dataset

Ensemble predictions - Min: 0.0081, Max: 0.8695
Mean: 0.3517, Std: 0.1514


In [29]:
# Create submission file
submission_df = pd.DataFrame({
    'id': test_ids,
    'accident_risk': ensemble_pred
})

submission_df.to_csv('submission.csv', index=False)
print("\nSubmission file saved to 'submission.csv'")
print(f"\nSubmission preview:\n{submission_df.head(10)}")
print(f"\nShape: {submission_df.shape}")


Submission file saved to 'submission.csv'

Submission preview:
       id  accident_risk
0  517754       0.312484
1  517755       0.131153
2  517756       0.200428
3  517757       0.334502
4  517758       0.388126
5  517759       0.445153
6  517760       0.272938
7  517761       0.186344
8  517762       0.408147
9  517763       0.334834

Shape: (172585, 2)
